In [1]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface
import spikeinterface.exporters as sexp
from spikeinterface.core import write_binary_recording
from pathlib import Path
import pickle
import MEArec as mr
from tabnanny import verbose
import spikeinterface as si
import numpy as np
from spikeinterface.core import get_template_extremum_channel
import scipy.spatial.distance
from scipy.sparse.csgraph import connected_components
import pickle
import pandas as pd
from utils_clique import (
    CliqueInfo,
    build_shank_cliques,
    neuron_inf_dict_to_dataframe,
    get_recording_clique,
    filter_neuron_inf_by_clique,
    filter_gt_detect_array_by_clique,
    prepare_training_data,
    train_autosort_model,
    build_sliding_cliques
)


In [2]:
recording, sorting = se.read_mearec("/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s.h5")
probe = recording.get_probe()
recording_recorded = spre.bandpass_filter(recording, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_recorded, freq=50)
recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")
probe.set_contact_ids(recording.channel_ids)

In [3]:
output_folder = '/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s'
cliques = build_sliding_cliques(
    probe,
    clique_size=49,
    min_size=25,
    min_overlap=18,
    target_groups=12,
)

clique_info = {
    'cliques': cliques,  # List of CliqueInfo objects
    'clique_params': {
        'clique_size': 49,
        'min_size': 25,
        'min_overlap': 18,
        'target_groups': 12,
    },
    'probe_df': probe.to_dataframe(),  # Probe dataframe for verification
}

clique_info_path = f'{output_folder}/clique_info.pkl'
with open(clique_info_path, 'wb') as f:
    pickle.dump(clique_info, f)

[INFO] Built 12 cliques (target 12)
       Clique 00: channels 192-12 (49 channels)
       Clique 01: channels 103-115 (49 channels)
       Clique 02: channels 303-123 (49 channels)
       Clique 03: channels 23-227 (49 channels)
       Clique 04: channels 31-43 (49 channels)
       Clique 05: channels 326-338 (49 channels)
       Clique 06: channels 334-154 (49 channels)
       Clique 07: channels 54-258 (49 channels)
       Clique 08: channels 254-74 (49 channels)
       Clique 09: channels 165-177 (49 channels)
       Clique 10: channels 173-185 (49 channels)
       Clique 11: channels 371-383 (49 channels)


In [ ]:
# 设置基本参数（这些参数应该与生成数据的脚本保持一致）
segment_duration_seconds = 600  # 每段600秒
n_segments = 6  # 总共6段

# 获取recording的采样率
sampling_frequency = recording_f.get_sampling_frequency()
total_num_samples = recording_f.get_num_samples()

# 计算每段的采样点数
segment_num_samples = int(segment_duration_seconds * sampling_frequency)

# 计算每个segment的采样点范围（用于提取recording片段）
segment_sample_ranges = {}  # {segment_idx: (start_sample, end_sample)}
for seg_idx in range(n_segments):
    start_sample = seg_idx * segment_num_samples
    # 最后一段可能不足600s，使用实际结束位置
    if seg_idx == n_segments - 1:
        end_sample = total_num_samples
    else:
        end_sample = (seg_idx + 1) * segment_num_samples
    
    segment_sample_ranges[seg_idx] = (start_sample, end_sample)

# 设置输出文件夹
combined_output_base = output_folder

# ============================================================
# 训练模型：对每个clique使用segment_0的数据训练5次
# ============================================================
print(f"\n{'='*60}")
print(f"开始训练模型：每个clique使用segment_0数据训练5次")
print(f"{'='*60}\n")

# 对每个clique进行训练
for clique in cliques:
    clique_id = clique.clique_id
    print(f"\n{'='*60}")
    print(f"训练 Clique {clique_id}")
    print(f"{'='*60}")
    
    # 读取segment_0的数据
    segment_idx = 0
    segment_data_folder = f'{combined_output_base}/clique_{clique_id}/segment_{segment_idx}'
    neuron_inf_path = f'{segment_data_folder}/neuron_inf.pickle'
    gt_detect_array_path = f'{segment_data_folder}/gt_detect_array.csv'
    
    if not os.path.exists(neuron_inf_path) or not os.path.exists(gt_detect_array_path):
        print(f"  警告: {segment_data_folder} 下没有找到数据文件，跳过")
        continue
    
    # 加载数据
    with open(neuron_inf_path, 'rb') as f:
        neuron_inf_dict = pickle.load(f)
    gt_detect_array = pd.read_csv(gt_detect_array_path)
    
    # 转换为DataFrame
    neuron_inf_segment = neuron_inf_dict_to_dataframe(neuron_inf_dict)
    
    print(f"  Segment {segment_idx} 数据:")
    print(f"    - Neurons: {len(neuron_inf_segment)}")
    print(f"    - Spikes: {len(gt_detect_array)}")
    
    if len(neuron_inf_segment) == 0 or len(gt_detect_array) == 0:
        print(f"  警告: Segment {segment_idx} 没有数据，跳过")
        continue
    
    # 获取segment_0对应的recording片段
    segment_start_sample, segment_end_sample = segment_sample_ranges[segment_idx]
    recording_segment = recording_f.frame_slice(
        start_frame=segment_start_sample,
        end_frame=segment_end_sample
    )
    
    # 获取recording_clique
    recording_clique = get_recording_clique(recording_segment, clique)
    print(f"  Recording clique channels: {len(recording_clique.get_channel_ids())}")
    
    # 保持gt_detect_array的time为采样点索引（不转换为秒）
    gt_detect_array_for_training = gt_detect_array.copy()
    
    # 将extremum_channel转换为字符串类型，以匹配recording_clique的channel IDs
    if 'extremum_channel' in gt_detect_array_for_training.columns:
        gt_detect_array_for_training['extremum_channel'] = gt_detect_array_for_training['extremum_channel'].astype(str)
    
    # 准备训练数据
    clique_save_dir = f'{combined_output_base}/clique_{clique_id}/segment_{segment_idx}'
    train_data_dir = prepare_training_data(
        recording_f=recording_clique,
        gt_detect_array=gt_detect_array_for_training,
        neuron_inf=neuron_inf_segment,
        save_dir=clique_save_dir,
        duration_seconds=segment_duration_seconds,  # 使用segment的时长（600秒）
        thr_min=2.5,
        thr_max=10,
        distance=3,
        wlen=5,
        prominence=15,
        left_sample=10,
        right_sample=20,
        max_firing_channel=None
    )
    
    # 训练模型（重复5次）
    n_channels = recording_clique.get_num_channels()
    n_repeats = 5
    
    for repeat_idx in range(1, n_repeats + 1):
        print(f"\n  ===== 重复训练 {repeat_idx}/{n_repeats} =====")
        model_save_dir = f'{clique_save_dir}/model_{repeat_idx}'
        
        autosort_model, training_log = train_autosort_model(
            train_data_dir=train_data_dir,
            model_save_dir=model_save_dir,
            n_channels=n_channels,
            left_sample=10,
            right_sample=20,
            epochs=20,
            batch_size=512,
            device=None,
            early_stopping=True,
            patience=5,
            min_delta=0.0,
            use_focal_loss=True,
            focal_gamma=2.0
        )
        
        print(f"  重复训练 {repeat_idx}/{n_repeats} 完成!")
    
    print(f"  Clique {clique_id} 所有重复训练完成!")

print("\n所有训练完成！")



开始训练模型：每个clique使用segment_0数据训练5次


训练 Clique 0
  Segment 0 数据:
    - Neurons: 14
    - Spikes: 37742
  Recording clique channels: 49
### 1. Threshold Detection
Sampling rate: 10000.0 Hz, Number of channels: 49
Recording total length: 6000000 samples (600.00 seconds)
Will process first 6000000 samples (600.00 seconds)
Data shape: (6000000, 49) (clique channels)
Using old detection method: extremum_channels
Using 11 valid channels from neuron extremum_channels
Building detect_array...
Number of detected spikes: 444097
去重: 移除了23612个spikes（保留幅值更大的channel上的spike）
去重前: 444097个spikes, 去重后: 420485个spikes

### 2. Load Ground Truth and Match
Building gt_array from gt_detect_array...
Filtered gt_detect_array: 37742 spikes (out of 37742 total)
Recording clique channel IDs (keys in probe_to_clique_index): [np.str_('193'), np.str_('1'), np.str_('289'), np.str_('97'), np.str_('2'), np.str_('194'), np.str_('290'), np.str_('98'), np.str_('195'), np.str_('3')]...
Sample extremum_channels from gt_detect

ValueError: gt_array is empty! This means no extremum_channels from gt_detect_array could be mapped to recording_clique channel IDs. Recording clique channel IDs: [np.str_('193'), np.str_('1'), np.str_('289'), np.str_('97'), np.str_('2'), np.str_('194'), np.str_('290'), np.str_('98'), np.str_('195'), np.str_('3'), np.str_('291'), np.str_('99'), np.str_('4'), np.str_('196'), np.str_('292'), np.str_('100'), np.str_('5'), np.str_('197'), np.str_('293'), np.str_('101'), np.str_('6'), np.str_('198'), np.str_('294'), np.str_('102'), np.str_('7'), np.str_('199'), np.str_('295'), np.str_('103'), np.str_('200'), np.str_('8'), np.str_('296'), np.str_('104'), np.str_('9'), np.str_('201'), np.str_('105'), np.str_('297'), np.str_('202'), np.str_('10'), np.str_('298'), np.str_('106'), np.str_('11'), np.str_('203'), np.str_('299'), np.str_('107'), np.str_('204'), np.str_('12'), np.str_('300'), np.str_('108'), np.str_('13')], Sample extremum_channels from gt_detect_array: [np.int64(1), np.int64(7), np